# DC Bike Safety Map — charts from a pipeline run

This notebook draws what the `ridescore` package produced. This notebook calculates
nothing: if a number here looks wrong, the fault is in `src/ridescore/`.

    uv run ridescore run --run-date 2026-08-05

`ridescore inspect` reports that run as numbers. This notebook draws the same run as charts
and a map.

The scoring that used to live in `notebooks/data_processing.ipynb` is now the `ridescore`
package. That older notebook is kept as a historical record and is no longer run.

See `docs/running-the-pipeline.md` for how a run is produced, and `docs/ported-defects.md`
for the mistakes the package reproduces faithfully and should not.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from ridescore import build, run_record

OUT = Path("../out")

segments = build.read(OUT, "road_segment")
scores = build.read(OUT, "ridescore_v1_scores")
crashes = build.read(OUT, "crashes")
run = run_record.read(OUT)

streets = segments.merge(scores, on="segment_id")
print(f"{len(segments):,} segments, {len(crashes):,} crashes")
print(f"run date {run['run_date']}, crash window {run['derived']['crash_window']}")
print(f"crash normalisation (95th percentile): {run['derived']['p95_crash_count']}")


## Level of traffic stress

In [ ]:
plt.hist(streets["lts_level"], bins=[0.5, 1.5, 2.5, 3.5, 4.5], rwidth=0.8)
plt.xticks([1, 2, 3, 4])
plt.xlabel("level of stress")
plt.ylabel("count")
plt.title("Histogram of LTS levels")
plt.show()


## What the network is made of

By length rather than by count, so a mile of arterial does not weigh the same as
a fifty-metre alley.


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 15))
total = streets["len"].sum()

for ax, column, title in [
    (axes[0, 0], "lts_level", "level of traffic stress"),
    (axes[0, 1], "speed_limit_raw", "posted speed limit"),
    (axes[1, 0], "num_lanes_raw", "number of lanes"),
    (axes[1, 1], "function", "road type"),
    (axes[2, 0], "bike_facility_type", "bike facility"),
    (axes[2, 1], "pavement_condition", "pavement condition"),
]:
    share = streets.groupby(column, dropna=False)["len"].sum() / total * 100
    share.plot(kind="bar", ax=ax, title=f"Percent of network by {title}", ylabel="percent")

plt.tight_layout()
plt.show()


## RideScore

In [ ]:
plt.hist(streets["ridescore_v1"], bins=20)
plt.xlabel("RideScore")
plt.ylabel("count")
plt.title("Histogram of RideScore v1")
plt.show()


## Injuries and fatalities in the window

The crash file is one row per crash. A crash can hurt more than one cyclist.


In [ ]:
counts = crashes[
    [
        "major_injuries_bicyclist",
        "minor_injuries_bicyclist",
        "unknown_injuries_bicyclist",
        "fatal_bicyclist",
    ]
].sum()
print(counts.to_string())
print(f"total: {counts.sum()}")


## The map

In [ ]:
ax = streets.plot(
    column="ridescore_v1",
    cmap="RdYlGn",
    linewidth=0.4,
    figsize=(12, 12),
    legend=True,
)
ax.set_axis_off()
ax.set_title("RideScore v1")
plt.show()
